# Практическая работа №4 — Loginom + GigaChat

Ноутбук проверяет `user_id`, получает TOP-10 товаров из Loginom и запрашивает персональную рекомендацию у GigaChat.

Создайте файл `.env` рядом с ноутбуком по образцу `.env.example`. Секреты не сохраняйте в ноутбуке.

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv('.env')
LOGINOM_URL = os.getenv(
    'LOGINOM_URL',
    'https://edu.loginom.dev/lgi/rest/instacart_ws_teacher/GetUserHistory',
)
LOGINOM_CHECK_URL = os.getenv('LOGINOM_CHECK_URL', '').strip()
LOGINOM_TIMEOUT = int(os.getenv('LOGINOM_TIMEOUT', '45'))
GIGA_KEY = os.getenv('GIGA_KEY', '').strip()
GIGA_MODEL = os.getenv('GIGA_MODEL', 'GigaChat-2')
if not GIGA_KEY:
    raise RuntimeError('Не найден GIGA_KEY. Добавьте ключ в .env.')

USER_ID = 1  # замените на ID пользователя для проверки
payload = {'Variables': {'user_id': USER_ID}}
rows = None  # при резервной проверке история будет получена заранее


## 1. Проверка существования пользователя
Если `LOGINOM_CHECK_URL` задан, вызывается отдельный сервис `CheckUserId`. Если он ещё не опубликован, используется временная проверка по наличию истории в опубликованном `GetUserHistory`.

In [ ]:
if LOGINOM_CHECK_URL:
    check_response = requests.post(
        LOGINOM_CHECK_URL, json=payload, timeout=LOGINOM_TIMEOUT
    )
    check_response.raise_for_status()
    check_data = check_response.json()
    if not isinstance(check_data, dict):
        raise ValueError('CheckUserId должен вернуть JSON-объект')

    check_rows = check_data.get('DataSet', {}).get('Rows', [])
    value = check_rows[0].get('user_exists') if check_rows and isinstance(check_rows[0], dict) else None
    if value is None and isinstance(check_data.get('Variables'), dict):
        value = check_data['Variables'].get('user_exists')
    if value is None:
        raise ValueError('В ответе CheckUserId отсутствует user_exists')
    user_exists = str(value).strip().casefold() in {'1', 'true', 'yes', 'да'}
    if not user_exists:
        raise ValueError(f'Пользователь user_id={USER_ID} не найден в Instacart')
    print(f'Пользователь user_id={USER_ID} найден сервисом CheckUserId')
else:
    print('LOGINOM_CHECK_URL не задан. Проверяю наличие истории через GetUserHistory.')
    history_response = requests.post(
        LOGINOM_URL, json=payload, timeout=LOGINOM_TIMEOUT
    )
    history_response.raise_for_status()
    history_data = history_response.json()
    rows = history_data.get('DataSet', {}).get('Rows', [])
    if not rows:
        raise ValueError(f'Пользователь user_id={USER_ID} не найден или история покупок пуста')
    print(f'Пользователь user_id={USER_ID} найден по истории покупок (резервная проверка)')


## 2. Получение истории покупок
Получаем максимум 10 наиболее часто покупаемых товаров. При резервной проверке запрос уже выполнен, повторно к Loginom не обращаемся.

In [ ]:
if rows is None:
    history_response = requests.post(
        LOGINOM_URL, json=payload, timeout=LOGINOM_TIMEOUT
    )
    history_response.raise_for_status()
    history_data = history_response.json()
    if not isinstance(history_data, dict):
        raise ValueError('GetUserHistory должен вернуть JSON-объект')
    rows = history_data.get('DataSet', {}).get('Rows', [])
if not isinstance(rows, list):
    raise ValueError('GetUserHistory: DataSet.Rows должен быть списком')
if not rows:
    raise ValueError('История покупок пользователя пуста')
rows = rows[:10]
df = pd.DataFrame(rows)
display(df)


## 3. Формирование контекста для модели

In [ ]:
lines = []
for i, row in enumerate(rows, start=1):
    lines.append(
        f"{i}. {row.get('product_name', 'Неизвестный товар')} — "
        f"отдел: {row.get('department_rus', row.get('department', 'не указан'))}, "
        f"категория: {row.get('aisle', 'не указана')}, "
        f"заказов: {row.get('order_count', 0)}"
    )
products_text = '\n'.join(lines)
print(products_text)


## 4. Запрос к GigaChat

In [ ]:
llm = GigaChat(
    credentials=GIGA_KEY,
    model=GIGA_MODEL,
    verify_ssl_certs=False,
    temperature=0.3,
    max_tokens=500,
)
system_prompt = '''Ты — персональный помощник покупателя в продуктовом онлайн-магазине.
Анализируй только переданную историю покупок, не выдумывай данные.
Обращайся к клиенту на «вы», пиши кратко — не более 5–6 предложений.'''
user_prompt = f'''Клиент user_id={USER_ID} чаще всего покупает:\n\n{products_text}\n\n
Начни с замеченных предпочтений и дай 1–2 дружеских совета.'''
answer = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
print(answer.content)


## 5. Проверка
Измените `USER_ID` на несуществующее значение и выполните ячейки заново. Отдельный `CheckUserId` должен остановить сценарий до получения истории. Если сервис пока не опубликован, резервный вариант остановится на пустом `GetUserHistory`. Для сдачи отдельный веб-сервис `CheckUserId` нужно опубликовать в Loginom и указать его URL в `.env`.